## 3 Way convo

In [ ]:
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
def husband_prompt(conversation_history):
    return f"""
    Your name is Mr. Ares. and you're the husband to Mrs. Ares. both you and your wife are trying to convince the car dealer to sell a car to your wife at a good price.

    this is the conversation history:
    {conversation_history}
    """

def wife_prompt(conversation_history):
    return f"""
    Your name is Mrs. Ares. and you're the wife to Mr. Ares. both you and your husband are trying to convince the car dealer to sell a car to you at a good price.

    this is the conversation history:
    {conversation_history}
    """

def car_dealer_prompt(conversation_history):
    return f"""
    Your name is Mr. Kenny the car dealer. and you're the car dealer. you're trying to sell a car to Mr. and Mrs. Ares.

    this is the conversation history:
    {conversation_history}
    """





system_prompt = """this is a conversation between a husband, wife and a car dealer. the husband and wife are trying to convince the car dealer to sell a car to the wife at a good price. the car dealer is trying to sell a car to the husband and wife at a good price.



Note: 
- response should be in json format
- output only a single JSON object with no other text, no markdown, and no code fences
- send only one response at a time
- only one person should be directed_to
- response must have the following keys: role, content, directed_to
- directed_to should be one of the following: husband, wife, dealer

response format:

    {
        "role": "husband or wife or dealer",
        "content": *response,
        "directed_to": "dealer or wife or husband"
    }

"""



In [ ]:
openai = OpenAI(base_url="http://localhost:11434/v1")

In [ ]:
!ollama pull deepseek-r1:1.5b

In [ ]:
import json
import re


def get_response(prompt):
    response = openai.chat.completions.create(
        model="deepseek-r1:1.5b",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ],
    )
    raw = (response.choices[0].message.content or "").strip()
    if not raw:
        raise ValueError("Model returned empty response")

    # Strip markdown code fence (e.g. ```json ... ```)
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\s*", "", raw)
        raw = re.sub(r"\s*```\s*$", "", raw)

    # Extract first {...} if there's extra text before/after
    match = re.search(r"\{.*\}", raw, re.DOTALL)
    if match:
        raw = match.group(0)

    try:
        message = json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Model did not return valid JSON. Raw response: {raw!r}") from e

    return message



In [ ]:
messages = []

max_messages = 10

while len(messages) < max_messages: 
    if not messages:
        prompt = husband_prompt(messages)
    else:
        last = messages[-1]
        directed_to = last.get("directed_to", "")
        if directed_to == "dealer":
            prompt = car_dealer_prompt(messages)
        elif directed_to == "wife":
            prompt = wife_prompt(messages)
        else:
            prompt = husband_prompt(messages)
    message = get_response(prompt)
    messages.append(message)
    print(message)